In [ ]:
# ============================================
# DEEPFAKE DETECTION - FULL TRAINING PIPELINE
# ============================================

# Step 1: Install & Setup
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "facenet-pytorch"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "scipy", "scikit-learn", "numpy"], check=True)

if not os.path.exists('/kaggle/working/project'):
    subprocess.run(["git", "clone", "https://github.com/Jokeera/deepfake-detection.git", "/kaggle/working/project"], check=True)

os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')

# Step 2: Find dataset
print("\n=== /kaggle/input/ contents ===")
if os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        full = os.path.join('/kaggle/input', item)
        print(f"  {item} ({'dir' if os.path.isdir(full) else 'file'})")
        if os.path.isdir(full):
            sub = os.listdir(full)[:5]
            print(f"    -> {sub}")
else:
    print("  /kaggle/input does NOT exist!")

DATA_ROOT = '/kaggle/input/preprocessed-dfdc02-16'
data_path = None

for root, dirs, files in os.walk(DATA_ROOT):
    if 'real' in dirs and 'fake' in dirs:
        data_path = root
        break

if data_path is None:
    if os.path.exists(DATA_ROOT):
        items = os.listdir(DATA_ROOT)
        print(f"Top items in dataset: {items[:10]}")
        for item in items:
            sub = os.path.join(DATA_ROOT, item)
            if os.path.isdir(sub):
                sub_dirs = os.listdir(sub)
                if 'real' in sub_dirs and 'fake' in sub_dirs:
                    data_path = sub
                    break
    if data_path is None:
        data_path = DATA_ROOT

real_path = os.path.join(data_path, 'real')
fake_path = os.path.join(data_path, 'fake')
real_count = len(os.listdir(real_path)) if os.path.exists(real_path) else 0
fake_count = len(os.listdir(fake_path)) if os.path.exists(fake_path) else 0
print(f'\nDataset: {data_path}')
print(f'Real: {real_count}, Fake: {fake_count}, Total: {real_count + fake_count}')

assert real_count > 0 and fake_count > 0, f"Dataset empty! Check structure of {DATA_ROOT}"

# Step 3: Run all experiments
from config import Config
from train import train
from utils import save_metrics
import time, json

EXPERIMENTS = [
    {'name': 'A1_full_model', 'model_type': 'full', 'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only', 'model_type': 'spatial', 'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal', 'fusion_type': 'adaptive'},
    {'name': 'A4_sequential', 'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

all_results = []
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*70}")
    print(f"[{i}/4] {exp['name']}")
    print(f"{'='*70}\n")
    cfg = Config()
    cfg.dataset_root = data_path
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']
    cfg.max_epochs = 30
    cfg.batch_size = 16
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.output_dir = './experiments'
    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f"\n✅ {exp['name']} done in {metrics['duration_min']} min")
        print(f"   AUC={test.get('auc',0):.4f} Acc={test.get('accuracy',0):.4f} F1={test.get('f1',0):.4f}")
    except Exception as e:
        print(f"\n❌ {exp['name']} FAILED: {e}")
        import traceback; traceback.print_exc()
        all_results.append({'experiment_name': exp['name'], 'status': 'failed', 'error': str(e)})
    save_metrics(all_results, './experiments/all_results.json')

# Step 4: Summary
print(f"\n{'='*70}")
print("RESULTS SUMMARY")
print(f"{'='*70}")
print(f"{'Experiment':<25} {'AUC':>8} {'Acc':>8} {'F1':>8} {'Time':>8}")
print('-' * 70)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        print(f"{r['experiment_name']:<25} {t.get('auc',0):>8.4f} {t.get('accuracy',0):>8.4f} {t.get('f1',0):>8.4f} {r.get('duration_min',0):>7.1f}m")
    else:
        print(f"{r['experiment_name']:<25} FAILED")

os.system("tar czf /kaggle/working/results.tar.gz experiments/ 2>/dev/null")
print(f"\n📦 results.tar.gz ready for download")
os.system("ls -lh /kaggle/working/results.tar.gz")